# Tuần 8 — chấm tầng 3 và tầng 4 trên `test`, lần cuối

Notebook này **không huấn luyện**. Nó nạp checkpoint BARTpho `train_20k` từ output
của kernel `dl-summarisevn-vit5` rồi sinh 2.000 bản tóm tắt của `test` cho hai cấu hình:

| Cấu hình | Là tầng nào |
|---|---|
| `--filter none` | tầng 3 — abstractive thuần |
| `--filter lexrank` | tầng 4 — lọc câu trước rồi viết lại |

## ĐỌC TRƯỚC KHI CHẠY

`test` dùng **đúng một lần** trong cả dự án. `vit5.py` từ chối `--eval-split test` trừ
khi có cờ `--cho-phep-test`, và notebook này truyền cờ đó — tức đây **là** lần chấm cuối.
Đừng chạy để thử đường ống: muốn thử thì đổi `EVAL_SPLIT` sang `val`.

Notebook `git pull` từ GitHub, nên **bản có `--cho-phep-test` phải được push trước**;
nếu không, kernel kéo về bản `vit5.py` cũ và chết ở bước từ chối.

## Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `GPU T4 x2` (notebook tự ghim còn một) |
| **Internet** | `On` |
| **Add data** | output của `dl-summarisevn-vit5` |

In [ ]:
# ==== CHI SUA O NAY ===================================================
EVAL_SPLIT = "test"                      # doi thanh "val" neu chi muon chay thu
NAME = "bartpho-syllable-train_20k"      # ten he thong trong bang ket qua
CKPT_GLOB = "/kaggle/input/**/vinai_bartpho-syllable_train_20k/final"

# Tang 3 (khong loc) va tang 4 (loc lexrank). Hai cau hinh nay du tra loi cau hoi 1 va 2
# tren tap cuoi; them lead_lexrank thi tach duoc "giu cau dau co ich khong" nhung ton
# them mot luot sinh 2.000 ban.
GRID = [
    {"filter": "none"},
    {"filter": "lexrank"},
]
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"
OUT = "/kaggle/working/test_final"


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi — `!lenh` loi khong nem ngoai le."""
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


if EVAL_SPLIT == "test":
    print("CANH BAO: day la lan cham CUOI CUNG tren `test`. Tap nay dung dung MOT lan.")
print(f"{len(GRID)} cau hinh tren tap {EVAL_SPLIT}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import urllib.request
import torch
print("GPU thay duoc:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Settings > Accelerator > GPU T4 x2.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("Khong ra duoc Internet. Settings > Internet > On.") from e
print("Moi truong: OK")

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
!git log --oneline -1

# Ban keo ve PHAI co co --cho-phep-test, khong thi den buoc sinh moi chet.
!grep -q 'cho-phep-test' src/models/vit5.py
check(_exit_code, "vit5.py co co --cho-phep-test")
print("cwd:", os.getcwd())

In [ ]:
import glob

found = sorted(glob.glob(CKPT_GLOB, recursive=True))
if not found:
    co_gi = sorted(glob.glob("/kaggle/input/*/*"))[:20]
    raise RuntimeError(
        f"Khong thay checkpoint khop {CKPT_GLOB}.\n"
        "Vao Add-ons > Add data > Your Work, them output cua notebook "
        "dl-summarisevn-vit5 (ban chay BARTpho train_20k).\n"
        f"Hien /kaggle/input co: {co_gi}"
    )
CKPT = found[0]
print("Checkpoint:", CKPT)
!ls -la {CKPT} | head -8

In [ ]:
import pathlib
import time

# Chup danh sach file ket qua CO SAN truoc khi chay, de o dong goi chi lay dung file
# cua lan nay.
co_san = {p.as_posix() for p in pathlib.Path("results").rglob("*.json")}

co_test = "--cho-phep-test " if EVAL_SPLIT == "test" else ""
t0 = time.time()
for i, g in enumerate(GRID, 1):
    cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --no-train "
           f"--model {CKPT} --name {NAME} --eval-split {EVAL_SPLIT} {co_test}"
           f"--filter {g['filter']} --out {OUT}")
    print(f"\n===== [{i}/{len(GRID)}] {g} =====")
    print(cmd)
    !{cmd}
    check(_exit_code, f"cau hinh {i} {g}")
print(f"\nXong {len(GRID)} cau hinh trong {(time.time() - t0) / 60:.1f} phut.")

In [ ]:
# Gom ket qua (khong gom trong so) thanh mot zip de tai ve tu tab Output.
import zipfile

picked = sorted(p for p in pathlib.Path("results").rglob("*.json")
                if p.as_posix() not in co_san and p.name.startswith(f"{NAME}_{EVAL_SPLIT}_"))
if not picked:
    raise RuntimeError("Khong thay file ket qua moi nao cua lan chay nay.")
zpath = f"/kaggle/working/ket_qua_test_final_{NAME}_{EVAL_SPLIT}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():86s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)

## Sau khi chạy

Tải `ket_qua_test_final_*.zip` từ tab Output, giải nén tại gốc repo. Hai bảng mới sẽ nằm
trong `results/tables/` với tên mang theo `test`, và `free_tag()` bảo đảm không đè bảng cũ.

Tầng 2 trên `test` chấm riêng tại máy (không cần GPU):

    ~/.venvs/torch/Scripts/python.exe src/models/phobert_select.py score \
        --model <thu muc final> --split test --cho-phep-test